In [1]:
%cd "/blue/cai6734/mombongu/ehr project"
!wget -r -N -c -np -nH --cut-dirs=2 -P mimic_demo https://physionet.org/files/mimic-iv-demo-omop/0.9/

/blue/cai6734/mombongu/ehr project
--2026-10-13 16:14:36--  https://physionet.org/files/mimic-iv-demo-omop/0.9/
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘mimic_demo/0.9/index.html’

0.9/index.html          [ <=>                ]     852  --.-KB/s    in 0.001s  

Last-modified header missing -- time-stamps turned off.
2026-10-13 16:14:37 (852 KB/s) - ‘mimic_demo/0.9/index.html’ saved [852]

Loading robots.txt; please ignore errors.
--2026-10-13 16:14:37--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 22 [text/plain]
Saving to: ‘mimic_demo/robots.txt’

robots.txt          100%[===================>]      22  --.-KB/s    in 0s      

2026-10-13 16:14:37 (41.1 MB/s) - ‘mimic_demo/robots.txt’ saved [22/22]

--2026-10-13 16:1

In [2]:
import os, glob, pandas as pd
root = "/blue/cai6734/mombongu/ehr project/mimic_demo"
for f in sorted(glob.glob(root + "/**/*.csv", recursive=True)):
    try:
        df = pd.read_csv(f, nrows=2)
        n = sum(1 for _ in open(f)) - 1
        print(f"\n{os.path.relpath(f, root)}  ({n} rows)")
        print("   ", list(df.columns))
    except Exception as e:
        print("\n", f, "-> could not read:", e)


0.9/1_omop_data_csv/2b_concept.csv  (3885 rows)
    ['concept_id', 'concept_name', 'domain_id', 'vocabulary_id', 'concept_class_id', 'standard_concept', 'concept_code', 'valid_start_DATE', 'valid_end_DATE', 'invalid_reason']

0.9/1_omop_data_csv/2b_concept_relationship.csv  (7716 rows)
    ['concept_id_1', 'concept_id_2', 'relationship_id', 'valid_start_DATE', 'valid_end_DATE', 'invalid_reason']

0.9/1_omop_data_csv/2b_vocabulary.csv  (27 rows)
    ['vocabulary_id', 'vocabulary_name', 'vocabulary_reference', 'vocabulary_version', 'vocabulary_concept_id']

0.9/1_omop_data_csv/attribute_definition.csv  (0 rows)
    ['attribute_definition_id', 'attribute_name', 'attribute_description', 'attribute_type_concept_id', 'attribute_syntax']

0.9/1_omop_data_csv/care_site.csv  (31 rows)
    ['care_site_id', 'care_site_name', 'place_of_service_concept_id', 'location_id', 'care_site_source_value', 'place_of_service_source_value']

0.9/1_omop_data_csv/cdm_source.csv  (1 rows)
    ['cdm_source_name'

In [3]:
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
concept = pd.read_csv(D + "2b_concept.csv", usecols=["concept_id", "concept_name"])
names = concept.drop_duplicates("concept_id").set_index("concept_id").concept_name

cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_concept_id", "condition_start_date"])
cond["name"] = cond.condition_concept_id.map(names)
print("CONDITIONS rows:", len(cond), "| named:", f"{cond.name.notna().mean():.0%}", "| concept_id 0:", f"{(cond.condition_concept_id == 0).mean():.0%}")
print(cond.groupby("name").person_id.nunique().sort_values(ascending=False).head(25).to_string())

meas = pd.read_csv(D + "measurement.csv", usecols=["person_id", "measurement_concept_id", "value_as_number", "value_source_value"])
meas["name"] = meas.measurement_concept_id.map(names)
print("\nMEASUREMENTS rows:", len(meas), "| named:", f"{meas.name.notna().mean():.0%}")
print(meas.groupby("name").person_id.nunique().sort_values(ascending=False).head(30).to_string())

person = pd.read_csv(D + "person.csv", usecols=["person_id", "gender_source_value", "year_of_birth"])
print("\nPERSON:", person.gender_source_value.value_counts().to_dict(), "| birth years", person.year_of_birth.min(), "to", person.year_of_birth.max())

CONDITIONS rows: 16441 | named: 0% | concept_id 0: 0%
Series([], )

MEASUREMENTS rows: 338550 | named: 0%
name
Stroke Volume Variation    1

PERSON: {'M': 57, 'F': 43} | birth years 2030 to 2149


In [2]:
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
c = pd.read_csv(D + "2b_concept.csv")
print("concept file: ids from", c.concept_id.min(), "to", c.concept_id.max())
print(c.vocabulary_id.value_counts().head(10).to_string())
print(c.domain_id.value_counts().head(10).to_string())
names = c.drop_duplicates("concept_id").set_index("concept_id").concept_name

for fname, std, src, srcval in [
    ("condition_occurrence.csv", "condition_concept_id", "condition_source_concept_id", "condition_source_value"),
    ("measurement.csv", "measurement_concept_id", "measurement_source_concept_id", "measurement_source_value"),
    ("procedure_occurrence.csv", "procedure_concept_id", "procedure_source_concept_id", "procedure_source_value"),
]:
    df = pd.read_csv(D + fname, usecols=[std, src, srcval])
    print(f"\n=== {fname}")
    print("standard id sample:", df[std].head(3).tolist(), "| named by concept file:", f"{df[std].map(names).notna().mean():.0%}")
    print("source id sample:  ", df[src].head(3).tolist(), "| named by concept file:", f"{df[src].map(names).notna().mean():.0%}")
    print("source value sample:", df[srcval].astype(str).head(8).tolist())

concept file: ids from 2000000000 to 2110000028
vocabulary_id
mimiciv_drug_ndc               1400
mimiciv_obs_drgcodes            916
mimiciv_micro_organism          364
mimiciv_meas_lab_loinc          251
mimiciv_meas_chart              169
mimiciv_proc_datetimeevents     129
mimiciv_proc_itemid             113
mimiciv_drug_route               96
mimiciv_micro_specimen           85
mimiciv_meas_unit                54
domain_id
Drug           1400
Observation    1332
Measurement     507
Procedure       246
Route            96
Specimen         81
Unit             58
Visit            53
Condition        34
Meas Value       30

=== condition_occurrence.csv
standard id sample: [196523, 436659, 435515] | named by concept file: 0%
source id sample:   [44824628, 44828816, 44822952] | named by concept file: 77%
source value sample: ['78791  ', '2809   ', '2761   ', '33829  ', '311    ', '27652  ', '34982  ', '25002  ']

=== measurement.csv
standard id sample: [3007913, 3012501, 3012501] | name

/tmp/ipykernel_2275084/598326637.py:14: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(D + fname, usecols=[std, src, srcval])


In [3]:
#see lab names
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
c = pd.read_csv(D + "2b_concept.csv")
names = c.drop_duplicates("concept_id").set_index("concept_id").concept_name

cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_source_concept_id", "condition_source_value", "condition_start_date"])
cond["name"] = cond.condition_source_concept_id.map(names)
print("CONDITIONS named:", f"{cond.name.notna().mean():.0%}")
print(cond.groupby("name").person_id.nunique().sort_values(ascending=False).head(30).to_string())
print("\nTop UNNAMED condition codes:")
print(cond[cond.name.isna()].condition_source_value.str.strip().value_counts().head(10).to_string())

meas = pd.read_csv(D + "measurement.csv", usecols=["person_id", "measurement_source_concept_id", "value_as_number", "value_source_value"])
meas["name"] = meas.measurement_source_concept_id.map(names)
print("\nLABS named:", f"{meas.name.notna().mean():.0%}")
print(meas.groupby("name").person_id.nunique().sort_values(ascending=False).head(40).to_string())
print("\nvalue_source_value sample:", meas.value_source_value.astype(str).head(8).tolist())

CONDITIONS named: 77%
name
SR (Sinus Rhythm)                      91
ST (Sinus Tachycardia)                 65
ETOH                                   58
SB (Sinus Bradycardia)                 28
AF (Atrial Fibrillation)               23
A Paced                                11
Post menopausal                         8
A Flut (Atrial Flutter)                 6
V Paced                                 5
1st AV (First degree AV Block)          4
AV Paced                                3
SA (Sinus Arrhythmia)                   3
SVT (Supra Ventricular Tachycardia)     3
LBBB (Left Bundle Branch Block)         2
JR (Junctional Rhythm)                  1

Top UNNAMED condition codes:
condition_source_value
4019     68
E785     57
2724     55
E039     47
42731    34
E1122    34
I2510    33
25000    33
I10      32
F329     32

LABS named: 73%
name
Heart Rhythm                                     100
Heart Rate                                       100
Edema Location                            

In [4]:
#cell count patients by disease group
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_source_value"], dtype=str)
code = cond.condition_source_value.str.strip().str.upper()

groups = {
 "Hypertension":        r"^(401|I10)",
 "Hyperlipidemia":      r"^(272|E78)",
 "Diabetes":            r"^(250|E1[0-3])",
 "Atrial fibrillation": r"^(4273|I48)",
 "Coronary disease":    r"^(414|I25)",
 "Heart failure":       r"^(428|I50)",
 "Chronic kidney dis.": r"^(585|N18)",
 "Acute kidney injury": r"^(584|N17)",
 "Anemia":              r"^(285|D5|D62|D63|D64)",
 "Pneumonia":           r"^(48[1-6]|J1[2-8])",
 "Sepsis":              r"^(038|A41|99591|R652)",
 "Hypothyroidism":      r"^(244|E03)",
 "Depression":          r"^(296[23]|F3[23])",
}
rows = {}
for g, rgx in groups.items():
    rows[g] = set(cond[code.str.contains(rgx, regex=True, na=False)].person_id)
for g, s in sorted(rows.items(), key=lambda x: -len(x[1])):
    print(f"{g:22s} {len(s):3d} patients")

per_pt = pd.Series({p: sum(p in s for s in rows.values()) for p in set().union(*rows.values())})
print("\nhow many of these groups does each patient have?")
print(per_pt.value_counts().sort_index().to_string())

Hypertension            53 patients
Hyperlipidemia          51 patients
Anemia                  45 patients
Diabetes                35 patients
Acute kidney injury     35 patients
Coronary disease        34 patients
Atrial fibrillation     31 patients
Chronic kidney dis.     26 patients
Heart failure           24 patients
Pneumonia               21 patients
Sepsis                  17 patients
Hypothyroidism          15 patients
Depression              10 patients

how many of these groups does each patient have?
1      9
2     11
3     15
4     14
5     15
6      2
7      9
8      6
9      5
10     1
12     1


/tmp/ipykernel_2275084/2855441018.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  rows[g] = set(cond[code.str.contains(rgx, regex=True, na=False)].person_id)


In [6]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv"
)

drug = pd.read_csv(DATA_DIR / "drug_exposure.csv", nrows=5)

print("Columns:")
print(drug.columns.tolist())

print("\nSample rows:")
print(drug.head().to_string())

Columns:
['drug_exposure_id', 'person_id', 'drug_concept_id', 'drug_exposure_start_date', 'drug_exposure_start_datetime', 'drug_exposure_end_date', 'drug_exposure_end_datetime', 'verbatim_end_date', 'drug_type_concept_id', 'stop_reason', 'refills', 'quantity', 'days_supply', 'sig', 'route_concept_id', 'lot_number', 'provider_id', 'visit_occurrence_id', 'visit_detail_id', 'drug_source_value', 'drug_source_concept_id', 'route_source_value', 'dose_unit_source_value']

Sample rows:
      drug_exposure_id            person_id  drug_concept_id drug_exposure_start_date drug_exposure_start_datetime drug_exposure_end_date drug_exposure_end_datetime  verbatim_end_date  drug_type_concept_id  stop_reason  refills  quantity  days_supply  sig  route_concept_id  lot_number  provider_id  visit_occurrence_id  visit_detail_id  drug_source_value  drug_source_concept_id route_source_value dose_unit_source_value
0   294884377115777655  1741351032930224901         40166274               2177-07-16          

In [7]:
DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv"
)

drug = pd.read_csv(DATA_DIR / "drug_exposure.csv")

print("=== DATA OVERVIEW ===")
print("Total medication records:", len(drug))
print("Unique patients:", drug["person_id"].nunique())
print("Unique drug concept IDs:", drug["drug_concept_id"].nunique())
print("Missing drug concept IDs:", drug["drug_concept_id"].isna().sum())
print("Missing source concept IDs:", drug["drug_source_concept_id"].isna().sum())

print("\n=== MOST COMMON DRUG CONCEPT IDs ===")
print(
    drug.groupby("drug_concept_id")
    .agg(
        record_count=("drug_exposure_id", "count"),
        patient_count=("person_id", "nunique")
    )
    .sort_values("patient_count", ascending=False)
    .head(15)
)

print("\n=== MOST COMMON SOURCE DRUG VALUES ===")
print(drug["drug_source_value"].value_counts().head(15))

print("\n=== RECORDS PER PATIENT ===")
print(
    drug.groupby("person_id")
    .size()
    .describe()
)

=== DATA OVERVIEW ===
Total medication records: 18229
Unique patients: 100
Unique drug concept IDs: 897
Missing drug concept IDs: 0
Missing source concept IDs: 0

=== MOST COMMON DRUG CONCEPT IDs ===
                 record_count  patient_count
drug_concept_id                             
40221385                  644             88
43011850                  296             85
0                         546             82
46275280                  794             82
19127213                  462             77
44784779                  196             77
938330                    183             75
1366754                   235             69
1560751                   183             67
40220357                  397             65
35605482                  156             65
19016865                  146             62
19020131                   99             61
19135374                  220             60
1127433                   133             60

=== MOST COMMON SOURCE DRUG VALUES

In [2]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9"
)

# Find available concept/vocabulary files
concept_files = [
    p for p in BASE_DIR.rglob("*.csv")
    if any(word in p.name.lower()
           for word in ["concept", "vocabulary", "relationship"])
]

print("=== CONCEPT / VOCABULARY FILES ===")
for p in concept_files:
    print(p)

# Check the most likely concept table
if concept_files:
    concept_path = next(
        (p for p in concept_files
         if p.name.lower() == "concept.csv"),
        concept_files[0]
    )

    concept = pd.read_csv(concept_path, low_memory=False)

    print("\n=== SELECTED FILE ===")
    print(concept_path)
    print("\nColumns:", concept.columns.tolist())
    print("\nSample rows:")
    print(concept.head(3).to_string(index=False))

    # Try matching the drug concept IDs to concept_id
    if "concept_id" in concept.columns:
        top_ids = drug["drug_concept_id"].value_counts().head(15).index
        matches = concept[
            concept["concept_id"].isin(top_ids)
        ]

        print("\n=== MATCHES FOR COMMON DRUG CONCEPT IDS ===")
        print(matches.to_string(index=False))
        print("\nMatched IDs:", matches["concept_id"].nunique(),
              "out of", len(top_ids))
else:
    print("No concept/vocabulary CSV files found.")


=== CONCEPT / VOCABULARY FILES ===
/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/2b_concept_relationship.csv
/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/2b_vocabulary.csv
/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/fact_relationship.csv
/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/2b_concept.csv

=== SELECTED FILE ===
/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/2b_concept_relationship.csv

Columns: ['concept_id_1', 'concept_id_2', 'relationship_id', 'valid_start_DATE', 'valid_end_DATE', 'invalid_reason']

Sample rows:
 concept_id_1  concept_id_2 relationship_id valid_start_DATE valid_end_DATE  invalid_reason
   2000003069       4022792         Maps to       1970-01-01     2099-12-31             NaN
   2000010663      40164921         Maps to       1970-01-01     2099-12-31             NaN
   2000020189      38001595         Maps to       1970-01-01     2099-12-31             NaN


In [9]:

from pathlib import Path

print("Current folder:", Path.cwd())
print("\nNotebook files:")
for file in Path.cwd().glob("*.ipynb"):
    print(file.name)

Current folder: /blue/cai6734/mombongu/ehr project

Notebook files:
MIMIC OMOP.ipynb
SYNTHEA TO OMOP.ipynb


In [12]:
!git clone https://github.com/mombongu/ehr-gen-ai-2026
%cd ehr-gen-ai-2026

Cloning into 'ehr-gen-ai-2026'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/blue/cai6734/mombongu/ehr project/ehr-gen-ai-2026


In [4]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv"
)

# Load the medication data and concept tables
drug = pd.read_csv(DATA_DIR / "drug_exposure.csv")
concept = pd.read_csv(DATA_DIR / "2b_concept.csv", low_memory=False)
relationships = pd.read_csv(
    DATA_DIR / "2b_concept_relationship.csv",
    low_memory=False
)

# Most common medication concept IDs
top_ids = drug["drug_concept_id"].value_counts().head(15).index

print("=== CONCEPT TABLE COLUMNS ===")
print(concept.columns.tolist())

print("\n=== DIRECT MATCHES IN CONCEPT TABLE ===")
direct = concept[concept["concept_id"].isin(top_ids)]
print(direct.to_string(index=False))
print("Direct matches:", direct["concept_id"].nunique(), "of", len(top_ids))

print("\n=== RELATIONSHIPS LEADING TO TOP DRUG IDS ===")
maps_to = relationships[
    (relationships["relationship_id"] == "Maps to")
    & (relationships["concept_id_2"].isin(top_ids))
]
print(maps_to.to_string(index=False))

print("\n=== RELATIONSHIPS STARTING FROM TOP DRUG IDS ===")
reverse_check = relationships[
    (relationships["relationship_id"] == "Maps to")
    & (relationships["concept_id_1"].isin(top_ids))
]
print(reverse_check.to_string(index=False))

if not maps_to.empty:
    source_ids = maps_to["concept_id_1"].unique()
    source_concepts = concept[
        concept["concept_id"].isin(source_ids)
    ]

    print("\n=== SOURCE CONCEPTS AND THEIR NAMES ===")
    print(source_concepts.to_string(index=False))


=== CONCEPT TABLE COLUMNS ===
['concept_id', 'concept_name', 'domain_id', 'vocabulary_id', 'concept_class_id', 'standard_concept', 'concept_code', 'valid_start_DATE', 'valid_end_DATE', 'invalid_reason']

=== DIRECT MATCHES IN CONCEPT TABLE ===
Empty DataFrame
Columns: [concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept, concept_code, valid_start_DATE, valid_end_DATE, invalid_reason]
Index: []
Direct matches: 0 of 15

=== RELATIONSHIPS LEADING TO TOP DRUG IDS ===
 concept_id_1  concept_id_2 relationship_id valid_start_DATE valid_end_DATE  invalid_reason
   2000011392      40221385         Maps to       1970-01-01     2099-12-31             NaN
   2000011361       1366754         Maps to       1970-01-01     2099-12-31             NaN
   2000011355      40221385         Maps to       1970-01-01     2099-12-31             NaN
   2000011384      46275280         Maps to       1970-01-01     2099-12-31             NaN
   2000011387      46275280         M

In [5]:

# IDs present in the included concept table
concept_ids = set(concept["concept_id"].dropna().astype("int64"))

# IDs that appear in the relationship table
mapped_standard_ids = set(
    relationships.loc[
        relationships["relationship_id"] == "Maps to",
        "concept_id_2"
    ].dropna().astype("int64")
)

print("=== CONCEPT COVERAGE ===")
print(
    "Drug concept IDs found in concept table:",
    drug["drug_concept_id"].isin(concept_ids).sum(),
    "of", len(drug)
)
print(
    "Source concept IDs found in concept table:",
    drug["drug_source_concept_id"].isin(concept_ids).sum(),
    "of", len(drug)
)
print(
    "Drug records whose standard IDs appear as mapping targets:",
    drug["drug_concept_id"].isin(mapped_standard_ids).sum(),
    "of", len(drug)
)

print("\n=== MOST COMMON SOURCE CONCEPT IDs ===")
print(drug["drug_source_concept_id"].value_counts().head(10))

print("\n=== MOST COMMON SOURCE DRUG VALUES ===")
print(drug["drug_source_value"].value_counts().head(10))

print("\n=== MAPPING COVERAGE BY DISTINCT STANDARD DRUG ID ===")
distinct_drug_ids = set(
    drug["drug_concept_id"].dropna().astype("int64")
)
print("Distinct standard drug IDs:", len(distinct_drug_ids))
print(
    "Distinct IDs present as mapping targets:",
    len(distinct_drug_ids & mapped_standard_ids)
)


=== CONCEPT COVERAGE ===
Drug concept IDs found in concept table: 0 of 18229
Source concept IDs found in concept table: 2431 of 18229
Drug records whose standard IDs appear as mapping targets: 3727 of 18229

=== MOST COMMON SOURCE CONCEPT IDs ===
drug_source_concept_id
0             546
2000011396    459
45317536      397
45249610      362
45144375      307
44955074      295
45060408      291
2000011355    249
2000011361    235
45197763      215
Name: count, dtype: int64

=== MOST COMMON SOURCE DRUG VALUES ===
drug_source_value
Sodium Chloride 0.9%  Flush 10 mL Syringe    459
00338004904                                  397
00409672924                                  362
00002751001                                  307
63323026201                                  295
00088222033                                  291
0.9% Sodium Chloride 100 mL Bag              249
Calcium Gluconate 2 g / 100 mL Premix Bag    235
00338011704                                  215
00338055318              

In [6]:

from itertools import combinations

# Exclude concept ID 0 because it is not a usable medication identifier.
usable_drugs = drug[drug["drug_concept_id"] != 0].copy()

# Build one set of distinct medication IDs per patient.
patient_profiles = (
    usable_drugs.groupby("person_id")["drug_concept_id"]
    .apply(lambda ids: set(ids.astype(int)))
    .to_dict()
)

print("Patients with usable medication profiles:", len(patient_profiles))
print(
    "Patients with at least one usable drug:",
    sum(bool(profile) for profile in patient_profiles.values())
)

# Compare every pair of patients using Jaccard similarity.
# Jaccard = shared unique drug IDs / total unique drug IDs across both patients.
results = []

for patient_a, patient_b in combinations(patient_profiles.keys(), 2):
    drugs_a = patient_profiles[patient_a]
    drugs_b = patient_profiles[patient_b]

    union = drugs_a | drugs_b
    if not union:
        continue

    shared = drugs_a & drugs_b
    score = len(shared) / len(union)

    results.append({
        "patient_a": patient_a,
        "patient_b": patient_b,
        "shared_drug_count": len(shared),
        "patient_a_drug_count": len(drugs_a),
        "patient_b_drug_count": len(drugs_b),
        "jaccard_similarity": round(score, 3),
        "shared_drug_ids": sorted(shared)
    })

similarity_df = pd.DataFrame(results)

similarity_df = similarity_df.sort_values(
    ["jaccard_similarity", "shared_drug_count"],
    ascending=False
)

print("\n=== TOP 10 MOST SIMILAR PATIENT PAIRS ===")
print(similarity_df.head(10).to_string(index=False))


Patients with usable medication profiles: 100
Patients with at least one usable drug: 100

=== TOP 10 MOST SIMILAR PATIENT PAIRS ===
           patient_a            patient_b  shared_drug_count  patient_a_drug_count  patient_b_drug_count  jaccard_similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          shared_drug_ids
-8492299714241840941  8805478484003283429                 36                    54                    38               0.643                                                                                 

In [7]:

# Get the highest-ranked patient pair
top_pair = similarity_df.iloc[0]

patient_a = top_pair["patient_a"]
patient_b = top_pair["patient_b"]

shared_ids = set(patient_profiles[patient_a]) & set(patient_profiles[patient_b])

print("=== TOP PATIENT PAIR ===")
print("Patient A:", patient_a)
print("Patient B:", patient_b)
print("Shared distinct drug IDs:", len(shared_ids))
print("Jaccard similarity:", top_pair["jaccard_similarity"])

# How common is each shared ID across all patients?
drug_patient_counts = (
    usable_drugs.groupby("drug_concept_id")["person_id"]
    .nunique()
)

shared_summary = pd.DataFrame({
    "drug_concept_id": sorted(shared_ids)
})

shared_summary["patients_with_drug"] = (
    shared_summary["drug_concept_id"]
    .map(drug_patient_counts)
)

shared_summary["percent_of_patients"] = (
    shared_summary["patients_with_drug"] / len(patient_profiles) * 100
).round(1)

print("\n=== MOST COMMON SHARED DRUG IDS ===")
print(
    shared_summary.sort_values(
        "patients_with_drug", ascending=False
    ).head(15).to_string(index=False)
)


=== TOP PATIENT PAIR ===
Patient A: -8492299714241840941
Patient B: 8805478484003283429
Shared distinct drug IDs: 36
Jaccard similarity: 0.643

=== MOST COMMON SHARED DRUG IDS ===
 drug_concept_id  patients_with_drug  percent_of_patients
        40221385                  88                 88.0
        46275280                  82                 82.0
        44784779                  77                 77.0
         1366754                  69                 69.0
         1560751                  67                 67.0
        19016865                  62                 62.0
        19020131                  61                 61.0
         1127433                  60                 60.0
        19135374                  60                 60.0
        36249737                  52                 52.0
        40167213                  47                 47.0
         1718752                  47                 47.0
        43647680                  47                 47.0
        

In [8]:

import math
from itertools import combinations

n_patients = len(patient_profiles)

# Number of distinct patients with each medication
drug_patient_counts = (
    usable_drugs.groupby("drug_concept_id")["person_id"]
    .nunique()
    .to_dict()
)

# Inverse document frequency (IDF):
# Common medications receive less weight.
drug_idf = {
    drug_id: math.log(
        (1 + n_patients) / (1 + patient_count)
    ) + 1
    for drug_id, patient_count in drug_patient_counts.items()
}

def weighted_jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 0.0

    intersection = set_a & set_b

    numerator = sum(drug_idf.get(d, 1.0) for d in intersection)
    denominator = sum(drug_idf.get(d, 1.0) for d in union)

    return numerator / denominator

weighted_results = []

for patient_a, patient_b in combinations(patient_profiles.keys(), 2):
    drugs_a = patient_profiles[patient_a]
    drugs_b = patient_profiles[patient_b]

    weighted_results.append({
        "patient_a": patient_a,
        "patient_b": patient_b,
        "shared_drug_count": len(drugs_a & drugs_b),
        "jaccard_similarity": round(
            len(drugs_a & drugs_b) / len(drugs_a | drugs_b), 3
        ) if drugs_a | drugs_b else 0.0,
        "weighted_jaccard": round(
            weighted_jaccard(drugs_a, drugs_b), 3
        )
    })

weighted_df = pd.DataFrame(weighted_results)

print("=== TOP 10 BY WEIGHTED JACCARD ===")
print(
    weighted_df.sort_values(
        "weighted_jaccard", ascending=False
    ).head(10).to_string(index=False)
)

print("\n=== TOP 10 BY ORIGINAL JACCARD ===")
print(
    weighted_df.sort_values(
        "jaccard_similarity", ascending=False
    ).head(10).to_string(index=False)
)


=== TOP 10 BY WEIGHTED JACCARD ===
           patient_a            patient_b  shared_drug_count  jaccard_similarity  weighted_jaccard
-3210373572193940939  8805478484003283429                 36               0.632             0.564
-8492299714241840941  8805478484003283429                 36               0.643             0.555
  817543968614095114  1484542834460282651                 55               0.611             0.543
-8492299714241840941 -3210373572193940939                 42               0.627             0.533
 6543064765547201849  8805478484003283429                 34               0.586             0.528
 1484542834460282651  7633320537823105326                 49               0.570             0.494
-3210373572193940939  6543064765547201849                 38               0.535             0.480
-8492299714241840941  6543064765547201849                 38               0.543             0.463
-3024822967781525875  1484542834460282651                 43              

In [12]:
%pip install streamlit
from pathlib import Path

import pandas as pd
import streamlit as st

DATA_DIR = Path(
    "/blue_cai6734/mombongu/ehr project/ehr-gen-ai-2026/0.9/1_omop_data_csv"
)

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide",
)

st.title("Clinical Patient Explorer")
st.caption(
    "Research prototype | Similar-patient retrieval "
    "and clinical record comparison"
)


@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)


person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")
visits = load_table("visit_occurrence.csv")


def patient_ids(df):
    if df.empty or "person_id" not in df.columns:
        return set()
    return set(df["person_id"].dropna())


if person.empty or conditions.empty:
    st.error(
        "Could not load person.csv or condition_occurrence.csv. "
        "Check DATA_DIR and the filenames."
    )
    st.stop()

all_ids = sorted(patient_ids(person), key=str)
aliases = {
    pid: f"Patient {i:03d}"
    for i, pid in enumerate(all_ids, start=1)
}

# Use standardized condition IDs when available.
# Fall back to source values only if a patient has no
# usable standardized condition IDs.
def condition_profile(pid):
    rows = conditions[conditions["person_id"] == pid]

    if "condition_concept_id" in rows.columns:
        ids = pd.to_numeric(
            rows["condition_concept_id"], errors="coerce"
        )
        usable = rows.loc[ids.notna() & (ids != 0)]
        concepts = set(
            pd.to_numeric(
                usable["condition_concept_id"],
                errors="coerce",
            ).dropna().astype(int)
        )
        if concepts:
            return concepts

    if "condition_source_value" in rows.columns:
        values = rows["condition_source_value"].dropna()
        return set(values.astype(str)) - {""}

    return set()


@st.cache_data
def build_profiles(ids):
    return {pid: condition_profile(pid) for pid in ids}


profiles = build_profiles(tuple(all_ids))


def jaccard(a, b):
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def display_values(df, pid, source_col, concept_col):
    if df.empty or "person_id" not in df.columns:
        return pd.DataFrame()

    rows = df[df["person_id"] == pid].copy()
    if rows.empty:
        return pd.DataFrame()

    # Prefer human-readable source values where available.
    if source_col in rows.columns:
        rows["Display"] = rows[source_col].fillna("").astype(str)
    elif concept_col in rows.columns:
        rows["Display"] = rows[concept_col].astype(str)
    else:
        return pd.DataFrame()

    if concept_col in rows.columns and source_col in rows.columns:
        missing = rows["Display"].isin(["", "nan", "None"])
        rows.loc[missing, "Display"] = (
            rows.loc[missing, concept_col].astype(str)
        )

    return rows[rows["Display"].ne("")]


selected_id = st.selectbox(
    "Select a patient",
    all_ids,
    format_func=lambda pid: aliases.get(pid, "Patient"),
)

selected_profile = profiles.get(selected_id, set())

ranked = []
for other_id, other_profile in profiles.items():
    if other_id == selected_id:
        continue

    score = jaccard(selected_profile, other_profile)
    ranked.append({
        "Patient": aliases.get(other_id, "Patient"),
        "ID": other_id,
        "Similarity": score,
        "Shared conditions": len(
            selected_profile & other_profile
        ),
    })

ranked.sort(key=lambda row: row["Similarity"], reverse=True)
top_matches = ranked[:5]

left, middle, right = st.columns(3)
left.metric("Recorded condition concepts", len(selected_profile))
middle.metric("Patients available", len(all_ids))
right.metric("Candidate matches", len(top_matches))

st.divider()
st.subheader("Selected patient's recorded conditions")

selected_conditions = display_values(
    conditions,
    selected_id,
    "condition_source_value",
    "condition_concept_id",
)

if selected_conditions.empty:
    st.info("No displayable condition values were found.")
else:
    st.dataframe(
        selected_conditions[["Display"]].drop_duplicates(),
        use_container_width=True,
        hide_index=True,
    )

st.subheader("Most similar patients")

if top_matches:
    results = pd.DataFrame(top_matches)
    results["Similarity"] = (
        results["Similarity"] * 100
    ).round(1).astype(str) + "%"

    st.dataframe(
        results[["Patient", "Similarity", "Shared conditions"]],
        use_container_width=True,
        hide_index=True,
    )

    match_options = [row["ID"] for row in top_matches]
    match_id = st.selectbox(
        "Inspect a matched patient",
        match_options,
        format_func=lambda pid: aliases.get(pid, "Patient"),
    )

    shared = selected_profile & profiles.get(match_id, set())
    st.write(f"Shared condition concepts: {len(shared)}")

    if shared:
        st.write(sorted(map(str, shared)))

    st.subheader("Recorded procedures for this match")
    procedure_rows = display_values(
        procedures,
        match_id,
        "procedure_source_value",
        "procedure_concept_id",
    )

    if procedure_rows.empty:
        st.info("No displayable procedure values found.")
    else:
        st.dataframe(
            procedure_rows[["Display"]].drop_duplicates(),
            use_container_width=True,
            hide_index=True,
        )

    st.subheader("Recorded measurements for this match")
    measurement_rows = (
        measurements[
            measurements["person_id"] == match_id
        ].copy()
        if not measurements.empty
        and "person_id" in measurements.columns
        else pd.DataFrame()
    )

    if measurement_rows.empty:
        st.info("No measurement records found.")
    else:
        columns = [
            c for c in [
                "measurement_date",
                "measurement_source_value",
                "measurement_concept_id",
                "value_as_number",
                "unit_source_value",
            ]
            if c in measurement_rows.columns
        ]
        st.dataframe(
            measurement_rows[columns].head(100),
            use_container_width=True,
            hide_index=True,
        )

else:
    st.info("No comparable patients found.")

st.caption(
    "Research prototype only. Similarity is based on condition "
    "overlap, not validated clinical equivalence or treatment advice. "
    "Unmapped codes and missing data can affect results."
)


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 17.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [streamlit]/8 [streamlit]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 0.4.10 requires pydantic<2.0,>=1.9, but you have pydantic 2.12.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2026-10-20 16:50:09.087 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:50:09.089 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:50:09.155 
  command:

    streamlit run /apps/jupyter/6.5.4/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-10-20 16:50:09.158 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:50:09.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:50:09.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:50:09.163 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored whe

DeltaGenerator()

In [13]:
import streamlit as st
print(st.__version__)

1.65.0


In [14]:

from pathlib import Path

import pandas as pd
import streamlit as st

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/"
    "mimic_demo/0.9/1_omop_data_csv"
)

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide",
)

st.title("Clinical Patient Explorer")
st.caption(
    "Research prototype | Similar-patient retrieval "
    "and clinical record comparison"
)


@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)


person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")
visits = load_table("visit_occurrence.csv")


def patient_ids(df):
    if df.empty or "person_id" not in df.columns:
        return set()
    return set(df["person_id"].dropna())


if person.empty or conditions.empty:
    st.error(
        "Could not load person.csv or condition_occurrence.csv. "
        "Check DATA_DIR and the filenames."
    )
    st.stop()

all_ids = sorted(patient_ids(person), key=str)
aliases = {
    pid: f"Patient {i:03d}"
    for i, pid in enumerate(all_ids, start=1)
}

# Use standardized condition IDs when available.
# Fall back to source values only if a patient has no
# usable standardized condition IDs.
def condition_profile(pid):
    rows = conditions[conditions["person_id"] == pid]

    if "condition_concept_id" in rows.columns:
        ids = pd.to_numeric(
            rows["condition_concept_id"], errors="coerce"
        )
        usable = rows.loc[ids.notna() & (ids != 0)]
        concepts = set(
            pd.to_numeric(
                usable["condition_concept_id"],
                errors="coerce",
            ).dropna().astype(int)
        )
        if concepts:
            return concepts

    if "condition_source_value" in rows.columns:
        values = rows["condition_source_value"].dropna()
        return set(values.astype(str)) - {""}

    return set()


@st.cache_data
def build_profiles(ids):
    return {pid: condition_profile(pid) for pid in ids}


profiles = build_profiles(tuple(all_ids))


def jaccard(a, b):
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def display_values(df, pid, source_col, concept_col):
    if df.empty or "person_id" not in df.columns:
        return pd.DataFrame()

    rows = df[df["person_id"] == pid].copy()
    if rows.empty:
        return pd.DataFrame()

    # Prefer human-readable source values where available.
    if source_col in rows.columns:
        rows["Display"] = rows[source_col].fillna("").astype(str)
    elif concept_col in rows.columns:
        rows["Display"] = rows[concept_col].astype(str)
    else:
        return pd.DataFrame()

    if concept_col in rows.columns and source_col in rows.columns:
        missing = rows["Display"].isin(["", "nan", "None"])
        rows.loc[missing, "Display"] = (
            rows.loc[missing, concept_col].astype(str)
        )

    return rows[rows["Display"].ne("")]


selected_id = st.selectbox(
    "Select a patient",
    all_ids,
    format_func=lambda pid: aliases.get(pid, "Patient"),
)

selected_profile = profiles.get(selected_id, set())

ranked = []
for other_id, other_profile in profiles.items():
    if other_id == selected_id:
        continue

    score = jaccard(selected_profile, other_profile)
    ranked.append({
        "Patient": aliases.get(other_id, "Patient"),
        "ID": other_id,
        "Similarity": score,
        "Shared conditions": len(
            selected_profile & other_profile
        ),
    })

ranked.sort(key=lambda row: row["Similarity"], reverse=True)
top_matches = ranked[:5]

left, middle, right = st.columns(3)
left.metric("Recorded condition concepts", len(selected_profile))
middle.metric("Patients available", len(all_ids))
right.metric("Candidate matches", len(top_matches))

st.divider()
st.subheader("Selected patient's recorded conditions")

selected_conditions = display_values(
    conditions,
    selected_id,
    "condition_source_value",
    "condition_concept_id",
)

if selected_conditions.empty:
    st.info("No displayable condition values were found.")
else:
    st.dataframe(
        selected_conditions[["Display"]].drop_duplicates(),
        use_container_width=True,
        hide_index=True,
    )

st.subheader("Most similar patients")

if top_matches:
    results = pd.DataFrame(top_matches)
    results["Similarity"] = (
        results["Similarity"] * 100
    ).round(1).astype(str) + "%"

    st.dataframe(
        results[["Patient", "Similarity", "Shared conditions"]],
        use_container_width=True,
        hide_index=True,
    )

    match_options = [row["ID"] for row in top_matches]
    match_id = st.selectbox(
        "Inspect a matched patient",
        match_options,
        format_func=lambda pid: aliases.get(pid, "Patient"),
    )

    shared = selected_profile & profiles.get(match_id, set())
    st.write(f"Shared condition concepts: {len(shared)}")

    if shared:
        st.write(sorted(map(str, shared)))

    st.subheader("Recorded procedures for this match")
    procedure_rows = display_values(
        procedures,
        match_id,
        "procedure_source_value",
        "procedure_concept_id",
    )

    if procedure_rows.empty:
        st.info("No displayable procedure values found.")
    else:
        st.dataframe(
            procedure_rows[["Display"]].drop_duplicates(),
            use_container_width=True,
            hide_index=True,
        )

    st.subheader("Recorded measurements for this match")
    measurement_rows = (
        measurements[
            measurements["person_id"] == match_id
        ].copy()
        if not measurements.empty
        and "person_id" in measurements.columns
        else pd.DataFrame()
    )

    if measurement_rows.empty:
        st.info("No measurement records found.")
    else:
        columns = [
            c for c in [
                "measurement_date",
                "measurement_source_value",
                "measurement_concept_id",
                "value_as_number",
                "unit_source_value",
            ]
            if c in measurement_rows.columns
        ]
        st.dataframe(
            measurement_rows[columns].head(100),
            use_container_width=True,
            hide_index=True,
        )

else:
    st.info("No comparable patients found.")

st.caption(
    "Research prototype only. Similarity is based on condition "
    "overlap, not validated clinical equivalence or treatment advice. "
    "Unmapped codes and missing data can affect results."
)


2026-10-20 16:51:50.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.704 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.706 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.707 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.710 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 16:51:50.711 No runtime found, using MemoryCacheStorageManager
2026-10-20 16:51:50.715 Thread 'MainThread':

DeltaGenerator()

In [15]:
from pathlib import Path

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/"
    "ehr-gen-ai-2026/0.9/1_omop_data_csv"
)

print("Directory exists:", DATA_DIR.exists())
print("Files found:", len(list(DATA_DIR.glob("*.csv"))))

print([f.name for f in DATA_DIR.glob("*.csv")][:10])

Directory exists: True
Files found: 32
['observation_period.csv', 'location.csv', '2b_concept.csv', 'drug_exposure.csv', 'provider.csv', 'device_exposure.csv', 'visit_occurrence.csv', 'note_nlp.csv', 'attribute_definition.csv', 'observation.csv']


In [16]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)

In [17]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [18]:
%run app.py

Exception: File `'app.py'` not found.

In [19]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Files here:", [p.name for p in Path.cwd().iterdir()][:30])

Current folder: /blue/cai6734/mombongu/ehr project/ehr-gen-ai-2026
Files here: ['README.md', 'MIMIC OMOP.ipynb', '.git', '0.9', '.ipynb_checkpoints']


In [20]:
from pathlib import Path

app_path = Path.cwd() / "app.py"

app_path.write_text(
    '''import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide"
)

DATA_DIR = Path(__file__).parent / "0.9" / "1_omop_data_csv"

@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
drugs = load_table("drug_exposure.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")

st.title("Clinical Patient Explorer")
st.write("Explore a patient and compare their records with other patients.")

if person.empty or conditions.empty:
    st.error(f"Could not find person.csv or condition_occurrence.csv in {DATA_DIR}")
    st.stop()

patient_ids = sorted(person["person_id"].dropna().unique(), key=str)

selected_id = st.selectbox(
    "Select a patient",
    patient_ids,
    format_func=lambda pid: f"Patient {pid}"
)

def get_profile(pid):
    rows = conditions[conditions["person_id"] == pid]
    if "condition_concept_id" not in rows.columns:
        return set()
    ids = pd.to_numeric(rows["condition_concept_id"], errors="coerce")
    return set(ids[ids.notna() & (ids != 0)].astype(int))

profiles = {pid: get_profile(pid) for pid in patient_ids}
selected_profile = profiles[selected_id]

def similarity(a, b):
    union = a | b

SyntaxError: incomplete input (1507565100.py, line 6)

In [21]:

%%writefile app.py
import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide"
)

DATA_DIR = Path(__file__).parent / "0.9" / "1_omop_data_csv"

@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
drugs = load_table("drug_exposure.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")

st.title("Clinical Patient Explorer")
st.write("Explore a patient and compare records across patients.")

if person.empty or conditions.empty:
    st.error(f"Could not find required files in {DATA_DIR}")
    st.stop()

patient_ids = sorted(person["person_id"].dropna().unique(), key=str)

def get_profile(pid):
    rows = conditions[conditions["person_id"] == pid]
    if "condition_concept_id" not in rows.columns:
        return set()
    ids = pd.to_numeric(rows["condition_concept_id"], errors="coerce")
    return set(ids[ids.notna() & (ids != 0)].astype(int))

profiles = {pid: get_profile(pid) for pid in patient_ids}

def similarity(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0

selected_id = st.selectbox(
    "Select a patient",
    patient_ids,
    format_func=lambda pid: f"Patient {pid}"
)

selected_profile = profiles[selected_id]
ranked = []

for pid in patient_ids:
    if pid == selected_id:
        continue
    other = profiles[pid]
    ranked.append({
        "Patient ID": pid,
        "Similarity": round(similarity(selected_profile, other), 3),
        "Shared condition concepts": len(selected_profile & other)
    })

ranked.sort(key=lambda x: x["Similarity"], reverse=True)

c1, c2, c3 = st.columns(3)
c1.metric("Patients", len(patient_ids))
c2.metric("Condition concepts", len(selected_profile))
c3.metric(
    "Drug exposure records",
    len(drugs[drugs["person_id"] == selected_id]) if not drugs.empty else 0
)

st.subheader("Most similar patients")
st.dataframe(pd.DataFrame(ranked[:10]), use_container_width=True, hide_index=True)

st.subheader("Medication records")
if not drugs.empty:
    rows = drugs[drugs["person_id"] == selected_id]
    cols = [c for c in [
        "drug_exposure_start_date", "drug_source_value",
        "drug_concept_id", "quantity"
    ] if c in rows.columns]
    st.dataframe(rows[cols].head(100), use_container_width=True, hide_index=True)
else:
    st.info("Drug exposure data not found.")

st.subheader("Procedure records")
if not procedures.empty:
    rows = procedures[procedures["person_id"] == selected_id]
    st.dataframe(rows.head(100), use_container_width=True, hide_index=True)
else:
    st.info("Procedure data not found.")

st.subheader("Measurement records")
if not measurements.empty:
    rows = measurements[measurements["person_id"] == selected_id]
    cols = [c for c in [
        "measurement_date", "measurement_source_value",
        "measurement_concept_id", "value_as_number", "unit_source_value"
    ] if c in rows.columns]
    st.dataframe(rows[cols].head(100), use_container_width=True, hide_index=True)
else:
    st.info("Measurement data not found.")

st.caption(
    "Research prototype. Similarity is based on condition overlap "
    "and is not a validated clinical matching method."
)


Writing app.py
